In [ ]:
# NLGCL Kaggle Notebook
# This notebook runs the NLGCL model on the Kaggle platform.
# It automatically sets up the environment, downloads the dataset, and runs training.

import os
import sys

# 1. Clone Repository (if not running inside it)
if not os.path.exists('NLGCL') and not os.path.exists('recbole_gnn'):
    !git clone https://github.com/Jinfeng-Xu/NLGCL.git
    os.chdir('NLGCL')
    print("Repository cloned and entered.")
elif os.path.basename(os.getcwd()) != 'NLGCL' and os.path.exists('NLGCL'):
    os.chdir('NLGCL')
    print("Entered NLGCL directory.")
else:
    print(f"Already in directory: {os.getcwd()}")

# Add current directory to sys.path to allow local imports (recbole_gnn)
sys.path.append(os.getcwd())

# 2. Install Dependencies
print("Installing dependencies...")
# Install packages from requirements.txt
!pip install -q -r requirements.txt
# Install PyTorch Geometric (compatible with Colab/Kaggle default Torch versions)
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

# 3. Dataset Preparation
# We use the helper script 'download_data.py' to fetch the Yelp dataset (~600MB)
# This handles downloading from RecBole's S3 bucket and extraction.
dataset_name = 'Yelp'
if not os.path.exists(f'dataset/{dataset_name}'):
    print(f"Dataset '{dataset_name}' not found. Downloading...")
    !python download_data.py
else:
    print(f"Dataset '{dataset_name}' found at dataset/{dataset_name}")

# 4. Run Training
from logging import getLogger
from recbole.utils import init_logger, init_seed, set_color
from recbole_gnn.config import Config
from recbole_gnn.utils import create_dataset, data_preparation, get_model, get_trainer
import torch

def run_single_model(args):
    # configurations initialization
    config = Config(
        model=args.model,
        dataset=args.dataset,
        config_file_list=args.config_file_list
    )
    
    # Ensure data_path is correctly set to local dataset/ folder
    if 'data_path' not in config:
        config['data_path'] = 'dataset/'

    init_seed(config['seed'], config['reproducibility'])
    init_logger(config)
    logger = getLogger()
    logger.info(config)

    # dataset filtering
    dataset = create_dataset(config)
    logger.info(dataset)

    # dataset splitting
    train_data, valid_data, test_data = data_preparation(config, dataset)

    # model loading and initialization
    # Note: Ensure the model class is available in recbole_gnn/model/general_recommender/nlgcl.py
    model = get_model(config['model'])(config, train_data.dataset).to(config['device'])
    logger.info(model)

    # trainer loading and initialization
    trainer = get_trainer(config['MODEL_TYPE'], config['model'])(config, model)

    # model training
    best_valid_score, best_valid_result = trainer.fit(
        train_data, valid_data, saved=True, show_progress=config['show_progress']
    )

    # model evaluation
    test_result = trainer.evaluate(test_data, load_best_model=True, show_progress=config['show_progress'])

    logger.info(set_color('best valid ', 'yellow') + f': {best_valid_result}')
    logger.info(set_color('test result', 'yellow') + f': {test_result}')


# Simulation of command-line arguments
class Args:
    def __init__(self):
        self.model = 'NLGCL'
        self.dataset = 'Yelp'
        self.config_files = ''
        self.config_file_list = ['properties/overall.yaml']

args = Args()

if torch.cuda.is_available():
    print(f"Running on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU. Training might be slow.")

try:
    if os.path.exists(f'dataset/{args.dataset}'):
         run_single_model(args)
    else:
         print(f"Error: Dataset {args.dataset} setup failed. Please check download.")
except Exception as e:
    import traceback
    traceback.print_exc()